# Team C — Cross-Model Pareto Analysis + Routing Simulation

**Runtime:** CPU is fine for this notebook (no model loading)  
**Estimated time:** 30-60 minutes  
**Prerequisite:** Teams A and B must have uploaded their result JSONs to the shared Google Drive folder.

## What this notebook does
1. Loads all calibration + profiling results from Google Drive
2. Runs Pareto dominance analysis across all model/precision configs
3. Generates 2D and 3D Pareto plots
4. Generates cross-model heatmaps
5. Runs uncertainty-based routing simulation
6. Produces publication-quality figures for the final report

## 0. Setup

In [ ]:
!pip install -q matplotlib scipy numpy
print('Done.')

In [ ]:
!git clone https://github.com/YOUR-ORG/uncertainty-aware-inference.git
%cd uncertainty-aware-inference
import sys
sys.path.insert(0, '.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Shared results folder — Teams A and B upload their JSONs here
DRIVE_BASE    = '/content/drive/MyDrive/uncertainty-inference'
CAL_DIRS      = [
    f'{DRIVE_BASE}/llama2_7b',
    f'{DRIVE_BASE}/mistral_7b',
    f'{DRIVE_BASE}/llama2_13b',
]
PROFILING_DIR = f'{DRIVE_BASE}/profiling'
OUTPUT_DIR    = f'{DRIVE_BASE}/analysis'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Check what's available
for d in CAL_DIRS + [PROFILING_DIR]:
    if os.path.exists(d):
        files = [f for f in os.listdir(d) if f.endswith('.json')]
        print(f'{d}: {len(files)} JSON files')
    else:
        print(f'MISSING: {d}')

## 1. Load All Results

In [ ]:
import json
import glob
from pathlib import Path
from src.analysis.pareto import ExperimentPoint

DATASET = 'arc_challenge'   # which dataset to use for Pareto analysis

points = []
for cal_dir in CAL_DIRS:
    if not os.path.exists(cal_dir):
        print(f'Skipping missing dir: {cal_dir}')
        continue

    for cal_file in sorted(glob.glob(f'{cal_dir}/*_{DATASET}_calibration.json')):
        tag       = Path(cal_file).stem.replace(f'_{DATASET}_calibration', '')
        prof_file = f'{PROFILING_DIR}/{tag}_profiling.json'

        with open(cal_file) as f:
            cal = json.load(f)

        if os.path.exists(prof_file):
            with open(prof_file) as f:
                prof = json.load(f)
        else:
            # Use placeholder profiling data if not available yet
            print(f'  [WARN] No profiling for {tag} — using placeholder')
            prof = {'tokens_per_sec': 0, 'gpu_mem_peak_gb': 0, 'latency_ms_mean': 0}

        points.append(ExperimentPoint.from_dicts(cal, prof))
        print(f'  Loaded: {tag}')

print(f'\nTotal: {len(points)} experiment points')

## 2. Pareto Analysis

In [ ]:
from src.analysis.pareto import pareto_front

pf = pareto_front(points)

print(f'Pareto front: {len(pf)} / {len(points)} configs are Pareto-optimal\n')
print(f'{"Model":15s}  {"Precision":12s}  {"ECE":>8s}  {"Accuracy":>8s}  {"tok/s":>8s}  {"Mem_GB":>8s}')
print('-' * 70)
for p in sorted(pf, key=lambda x: x.ece):
    print(f'{p.model_name:15s}  {p.precision:12s}  '
          f'{p.ece:8.4f}  {p.accuracy:8.4f}  '
          f'{p.tokens_per_sec:8.1f}  {p.gpu_mem_gb:8.2f}')

In [ ]:
from src.analysis.pareto import plot_pareto_2d, plot_pareto_3d
import matplotlib.pyplot as plt

# 2D Pareto: ECE vs Accuracy
fig1 = plot_pareto_2d(
    points, pf,
    x_attr='ece', y_attr='accuracy',
    x_label='ECE (↓ better)', y_label='Accuracy (↑ better)',
    save_path=f'{OUTPUT_DIR}/pareto_ece_vs_accuracy.png',
    title='Pareto: Calibration vs Accuracy\n(★ = Pareto-optimal)'
)
plt.show()

# 2D Pareto: Latency vs Accuracy
if all(p.latency_ms > 0 for p in points):
    fig2 = plot_pareto_2d(
        points, pf,
        x_attr='latency_ms', y_attr='accuracy',
        x_label='Latency (ms/token)', y_label='Accuracy (↑ better)',
        save_path=f'{OUTPUT_DIR}/pareto_latency_vs_accuracy.png',
    )
    plt.show()

In [ ]:
# 3D Pareto
if all(p.latency_ms > 0 for p in points):
    fig3d = plot_pareto_3d(points, pf, save_path=f'{OUTPUT_DIR}/pareto_3d.png')
    plt.show()
    print('3D Pareto plot saved.')

## 3. Cross-Model Heatmaps

In [ ]:
from src.analysis.pareto import plot_cross_model_heatmap

for metric in ['ece', 'brier', 'accuracy']:
    try:
        fig = plot_cross_model_heatmap(
            points, metric,
            save_path=f'{OUTPUT_DIR}/heatmap_{metric}.png'
        )
        plt.show()
    except Exception as e:
        print(f'Heatmap failed for {metric}: {e}')

## 4. Routing Simulation

In [ ]:
import numpy as np
from src.analysis.pareto import simulate_routing, find_optimal_threshold, plot_routing

# Group by model — get FP16 baseline per model
fp16_by_model = {p.model_name: p for p in points if p.precision == 'FP16'}

routing_summary = {}

for p in points:
    if p.precision == 'FP16':
        continue
    fp16 = fp16_by_model.get(p.model_name)
    if fp16 is None:
        continue

    # Simulate confidence arrays
    # (In production: load real per-sample confidences saved during calibration eval)
    rng = np.random.default_rng(42)
    N   = 1000
    confidences   = rng.beta(5, 2, N).clip(0.01, 0.99)
    correct_quant = (rng.uniform(size=N) < p.accuracy).astype(float)
    correct_fp16  = (rng.uniform(size=N) < fp16.accuracy).astype(float)

    routing = simulate_routing(
        confidences=confidences, correct_quant=correct_quant,
        correct_fp16=correct_fp16,
        ece_quant=p.ece, ece_fp16=fp16.ece,
    )

    optimal = find_optimal_threshold(
        routing, fp16_acc=fp16.accuracy, fp16_ece=fp16.ece,
        acc_tolerance=0.01, ece_tolerance=0.005,
    )

    tag = f'{p.model_name}_{p.precision}'.replace(' ', '_')
    plot_routing(
        routing, fp16.accuracy, fp16.ece, optimal,
        label=f'{p.model_name} {p.precision}',
        save_path=f'{OUTPUT_DIR}/routing_{tag}.png',
    )

    if optimal:
        routing_summary[tag] = {
            'optimal_threshold'  : optimal.threshold,
            'cost_saving_pct'    : optimal.cost_saving_pct,
            'frac_cheap'         : optimal.frac_cheap,
            'acc_drop'           : fp16.accuracy - optimal.effective_acc,
            'ece_increase'       : optimal.effective_ece - fp16.ece,
        }
        print(f'{p.model_name:15s} {p.precision:12s} | '
              f'θ*={optimal.threshold:.2f}  '
              f'saving={optimal.cost_saving_pct:.1f}%  '
              f'frac_cheap={optimal.frac_cheap:.1%}')
    else:
        print(f'{p.model_name:15s} {p.precision:12s} | No threshold meets quality constraints')

## 5. Final Report Figures

In [ ]:
# Publication-quality calibration degradation summary
import matplotlib.pyplot as plt
import numpy as np

# ECE degradation vs FP16 per precision, averaged across models
precisions  = ['GPTQ_INT8', 'GPTQ_INT4', 'AWQ_INT4', 'NF4']
models      = list(fp16_by_model.keys())

fig, ax = plt.subplots(figsize=(10, 6))
x       = np.arange(len(precisions))
width   = 0.8 / len(models)
colors  = plt.cm.tab10(np.linspace(0, 1, len(models)))

for i, (model_name, color) in enumerate(zip(models, colors)):
    fp16 = fp16_by_model.get(model_name)
    if fp16 is None:
        continue

    deltas = []
    for prec in precisions:
        match = next((p for p in points if p.model_name == model_name and p.precision == prec), None)
        deltas.append((match.ece - fp16.ece) if match else 0.0)

    offset = (i - len(models)/2 + 0.5) * width
    ax.bar(x + offset, deltas, width=width*0.9, color=color, label=model_name)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xticks(x)
ax.set_xticklabels(precisions, fontsize=11)
ax.set_ylabel('ΔECE vs FP16 (positive = worse calibration)', fontsize=11)
ax.set_title('Calibration Degradation from Quantization\n(Δ ECE vs FP16 baseline)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

fig.tight_layout()
fig.savefig(f'{OUTPUT_DIR}/ece_degradation_summary.png', dpi=200, bbox_inches='tight')
plt.show()
print('ECE degradation figure saved — ready for paper.')

In [ ]:
# Save all analysis outputs
with open(f'{OUTPUT_DIR}/pareto_front.json', 'w') as f:
    json.dump([p.to_dict() for p in pf], f, indent=2)

with open(f'{OUTPUT_DIR}/routing_summary.json', 'w') as f:
    json.dump(routing_summary, f, indent=2)

print('All Team C outputs saved to Google Drive.')
print(f'\nFiles in {OUTPUT_DIR}:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f'  {f}')